# 门将扑救率分析：晋级 vs 出局球队 (FIFA World Cup 2026)

**分析问题：** 在2026年国际足联世界杯中，成功晋级淘汰赛阶段（16强及以上）球队的主力门将，其扑救率是否显著高于在小组赛阶段即遭淘汰球队的主力门将？

本 Notebook 包含两部分：
1. **数据抓取**：从 FBref 抓取门将统计表和赛程表，生成分析用的 CSV
2. **统计分析**：描述性统计、置信区间、双样本 t 检验（Welch's t-test）、可视化

> 注意：Part 1 需要联网才能跑通（抓取 fbref.com），如果你已经有现成的 CSV，可以跳过 Part 1，直接从 Part 2 开始。


## Part 1: 数据抓取（FBref）

In [ ]:
import re
import time
import requests
import pandas as pd
from io import StringIO

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; CDU-student-project/1.0)"}

def get_fbref_tables(url):
    """抓取一个 FBref 页面，返回所有表格（包括藏在 HTML 注释里的）"""
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    html = resp.text

    tables = []
    try:
        tables += pd.read_html(StringIO(html))
    except ValueError:
        pass

    # FBref 把很多表格藏在 <!-- --> 注释里，需要单独取出来解析
    comments = re.findall(r"<!--(.*?)-->", html, re.DOTALL)
    for c in comments:
        if "<table" in c:
            try:
                tables += pd.read_html(StringIO(c))
            except ValueError:
                pass
    return tables


In [ ]:
# 1.1 门将统计表 (Squad Goalkeeping)
gk_url = "https://fbref.com/en/comps/1/2026/keepers/2026-World-Cup-Stats"
gk_tables = get_fbref_tables(gk_url)

squad_gk = gk_tables[0].copy()
squad_gk.columns = squad_gk.columns.get_level_values(-1)   # 展开多层表头
squad_gk = squad_gk[squad_gk["Squad"] != "Squad"]           # 去掉重复表头行
squad_gk = squad_gk[["Squad", "GA", "Saves", "Save%"]].dropna()
squad_gk[["GA", "Saves", "Save%"]] = squad_gk[["GA", "Saves", "Save%"]].apply(
    pd.to_numeric, errors="coerce"
)
squad_gk.head()


In [ ]:
time.sleep(4)  # 礼貌延迟，避免被限流

# 1.2 赛程表，用来判断哪些队打进了淘汰赛
fixtures_url = "https://fbref.com/en/comps/1/2026/schedule/2026-World-Cup-Scores-and-Fixtures"
fixtures_tables = get_fbref_tables(fixtures_url)
fixtures = fixtures_tables[0].copy()

# 'Round' 列标出每场比赛所属阶段，比如 "Group Stage" / "Round of 32" / ...
knockout_matches = fixtures[fixtures["Round"] != "Group Stage"]
knockout_teams = set(knockout_matches["Home"]).union(set(knockout_matches["Away"]))

squad_gk["advanced"] = squad_gk["Squad"].isin(knockout_teams)
squad_gk.head()


In [ ]:
# 1.3 保存成分析用的 CSV
squad_gk.to_csv("goalkeeper_advanced_vs_eliminated.csv", index=False)
print(f"晋级组 (advanced=True):  {(squad_gk['advanced']==True).sum()} 支球队")
print(f"出局组 (advanced=False): {(squad_gk['advanced']==False).sum()} 支球队")
squad_gk


**核对提示：** 晋级组 + 出局组的球队数量应该正好是 32 + 16 = 48。如果对不上，检查一下队名拼写是否一致（比如 "Korea Republic" vs "South Korea"），或者是否有球队被 `isin()` 漏判。

## Part 2: 统计分析

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

df = pd.read_csv("goalkeeper_advanced_vs_eliminated.csv")
df["Save%"] = pd.to_numeric(df["Save%"], errors="coerce")
df = df.dropna(subset=["Save%"])

# 可选：剔除出场时间极短的替补门将（如果原始数据里带 Min/90s 列，建议加这一步）
# df = df[df["90s"] >= 1.0]

adv = df.loc[df["advanced"] == True, "Save%"]
elim = df.loc[df["advanced"] == False, "Save%"]

print(f"晋级组样本量 n_A = {len(adv)}")
print(f"出局组样本量 n_B = {len(elim)}")


### 2.1 描述性统计

In [ ]:
def describe(series, label):
    print(f"--- {label} 描述性统计 ---")
    print(f"均值 (Mean):     {series.mean():.2f}")
    print(f"中位数 (Median): {series.median():.2f}")
    print(f"标准差 (Std):    {series.std(ddof=1):.2f}")
    print(f"最小值 (Min):    {series.min():.2f}")
    print(f"最大值 (Max):    {series.max():.2f}\n")

describe(adv, "晋级组 (Advanced)")
describe(elim, "出局组 (Eliminated)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].boxplot([adv, elim], tick_labels=["Advanced", "Eliminated"])
axes[0].set_title("Goalkeeper Save% by Team Outcome")
axes[0].set_ylabel("Save %")

axes[1].hist(adv, bins=8, alpha=0.6, label="Advanced")
axes[1].hist(elim, bins=8, alpha=0.6, label="Eliminated")
axes[1].set_title("Distribution of Save%")
axes[1].set_xlabel("Save %")
axes[1].legend()

plt.tight_layout()
plt.savefig("save_pct_distribution.png", dpi=150)
plt.show()


### 2.2 置信区间 (95% CI)

In [ ]:
mean_adv = adv.mean()
sem_adv = stats.sem(adv)
ci_adv = stats.t.interval(confidence=0.95, df=len(adv) - 1, loc=mean_adv, scale=sem_adv)
print(f"晋级组扑救率均值 95% CI: {mean_adv:.2f} -> [{ci_adv[0]:.2f}, {ci_adv[1]:.2f}]")

mean_elim = elim.mean()
sem_elim = stats.sem(elim)
ci_elim = stats.t.interval(confidence=0.95, df=len(elim) - 1, loc=mean_elim, scale=sem_elim)
print(f"出局组扑救率均值 95% CI: {mean_elim:.2f} -> [{ci_elim[0]:.2f}, {ci_elim[1]:.2f}]")


### 2.3 前置假设检验（正态性 & 方差齐性）

In [ ]:
shapiro_adv = stats.shapiro(adv)
shapiro_elim = stats.shapiro(elim)
print(f"Shapiro-Wilk (晋级组): W={shapiro_adv.statistic:.3f}, p={shapiro_adv.pvalue:.3f}")
print(f"Shapiro-Wilk (出局组): W={shapiro_elim.statistic:.3f}, p={shapiro_elim.pvalue:.3f}")

levene = stats.levene(adv, elim)
print(f"Levene 方差齐性检验: statistic={levene.statistic:.3f}, p={levene.pvalue:.3f}")
# p < 0.05 说明两组方差不齐 -> 使用 Welch's t-test (equal_var=False)


### 2.4 双样本 t 检验 (Welch's t-test)

In [ ]:
t_stat, p_value = stats.ttest_ind(adv, elim, equal_var=False, alternative="greater")
# alternative="greater" 对应单尾假设 H1: 晋级组均值 > 出局组均值
# 想用双尾检验就把 alternative 改成 "two-sided"

print("H0: mu_晋级组 = mu_出局组")
print("H1: mu_晋级组 > mu_出局组  (单尾)")
print(f"t 统计量 = {t_stat:.3f}")
print(f"p 值     = {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"结论: p < {alpha}，拒绝 H0。有统计学证据支持晋级组门将扑救率显著更高。")
else:
    print(f"结论: p >= {alpha}，不能拒绝 H0。没有充分证据表明两组扑救率存在显著差异。")

pooled_std = np.sqrt(((len(adv) - 1) * adv.std(ddof=1) ** 2 +
                       (len(elim) - 1) * elim.std(ddof=1) ** 2) /
                      (len(adv) + len(elim) - 2))
cohens_d = (mean_adv - mean_elim) / pooled_std
print(f"\n效应量 Cohen's d = {cohens_d:.3f}")
